# Quality - Validate training_dataset_v0

Valida claves, calidad global y targets futuros de la tabla Gold v0.

In [ ]:
from pyspark.sql import functions as F

LEVEL_TABLE = 'weather.silver.river_levels_daily'
TEMP_TABLE = 'weather.silver.temperature_daily'
RAIN_TABLE = 'weather.silver.rainfall_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
GOLD_TABLE = 'weather.gold.training_dataset_v0'
PUNTO_PREDICCION = 'ana_74100000'
TARGET_STATION = '74100000'

In [ ]:
def assert_table_has_rows(table_name, filter_expr=None):
    df = spark.table(table_name)
    if filter_expr is not None:
        df = df.filter(filter_expr)
    row_count = df.count()
    print(f'{table_name}: {row_count} rows')
    if row_count == 0:
        raise ValueError(f'{table_name} has no rows')


def assert_unique(table_name, key_cols, filter_expr=None):
    df = spark.table(table_name)
    if filter_expr is not None:
        df = df.filter(filter_expr)
    duplicates = df.groupBy(*key_cols).count().filter(F.col('count') > 1)
    duplicate_count = duplicates.count()
    print(f'{table_name} duplicate keys on {key_cols}: {duplicate_count}')
    if duplicate_count > 0:
        duplicates.show(20, truncate=False)
        raise ValueError(f'{table_name} has duplicate keys')


def latest_quality(source_table, attribute_name):
    rows = (
        spark.table(QUALITY_TABLE)
        .filter(F.col('source_table') == F.lit(source_table))
        .filter(F.col('attribute_name') == F.lit(attribute_name))
        .filter(F.col('grain') == F.lit('global_source_daily'))
        .orderBy(F.col('evaluated_at').desc_nulls_last())
        .limit(1)
        .collect()
    )
    if not rows:
        raise ValueError(f'Missing attribute_quality for {source_table}.{attribute_name}')
    return rows[0]


def assert_future_target(horizon_days, target_col):
    base = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).alias('base')
    future = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).alias('future')
    mismatches = (
        base.join(
            future,
            F.date_add(F.col('base.fecha'), horizon_days) == F.col('future.fecha'),
            'inner',
        )
        .filter(F.col(f'base.{target_col}').isNotNull())
        .filter(F.col('future.nivel_rio_actual_m').isNotNull())
        .filter(F.abs(F.col(f'base.{target_col}') - F.col('future.nivel_rio_actual_m')) > F.lit(0.000001))
    )
    mismatch_count = mismatches.count()
    print(f'{target_col} mismatches: {mismatch_count}')
    if mismatch_count > 0:
        mismatches.select('base.fecha', F.col(f'base.{target_col}'), F.col('future.fecha'), F.col('future.nivel_rio_actual_m')).show(20, truncate=False)
        raise ValueError(f'{target_col} does not match future level')

In [ ]:
for table_name in [LEVEL_TABLE, TEMP_TABLE, QUALITY_TABLE, GOLD_TABLE]:
    spark.sql(f'DESCRIBE {table_name}').show(truncate=False)

assert_table_has_rows(LEVEL_TABLE, F.col('codigoestacao') == F.lit(TARGET_STATION))
assert_table_has_rows(TEMP_TABLE)
assert_table_has_rows(QUALITY_TABLE)
assert_table_has_rows(GOLD_TABLE, F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

assert_unique(LEVEL_TABLE, ['fecha', 'codigoestacao'], F.col('codigoestacao') == F.lit(TARGET_STATION))
assert_unique(TEMP_TABLE, ['fecha', 'estacion_id'])
assert_unique(GOLD_TABLE, ['fecha', 'punto_prediccion'], F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

for source_table, attribute_name in [
    (LEVEL_TABLE, 'nivel_media_cm'),
    (TEMP_TABLE, 'temp_media_c'),
    (TEMP_TABLE, 'temp_min_c'),
    (TEMP_TABLE, 'temp_max_c'),
    (RAIN_TABLE, 'lluvia_acumulada_mm'),
]:
    quality = latest_quality(source_table, attribute_name)
    print(f"quality {source_table}.{attribute_name}: missing_pct={quality['missing_pct']}, is_usable={quality['is_usable']}")

gold = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

# R8 (Decision 019): no hay porton binario para lluvia. Se valida que la cobertura
# publicada sea coherente, en vez de un missing_pct global que ya no bloquea nada.
bad_coverage_rows = gold.filter(
    (F.col('lluvia_agregado_alta_frontera_cobertura_pct') < 0) | (F.col('lluvia_agregado_alta_frontera_cobertura_pct') > 1)
).count()
print(f'rows with lluvia_agregado_alta_frontera_cobertura_pct out of [0,1]: {bad_coverage_rows}')
if bad_coverage_rows > 0:
    raise ValueError('lluvia_agregado_alta_frontera_cobertura_pct out of the [0,1] range')

stale_gate_rows = gold.filter(F.col('lluvia_is_usable').isNotNull()).count()
print(f'rows with lluvia_is_usable populated (deprecated column, should stay NULL): {stale_gate_rows}')
if stale_gate_rows > 0:
    raise ValueError('lluvia_is_usable should be NULL under R8; the old all-or-nothing gate is deprecated')

# R8 aplicado a temperatura (Fase 3, Decision 025): mismo assert de cobertura que lluvia.
bad_temp_coverage_rows = gold.filter(
    (F.col('temp_agregado_alta_frontera_cobertura_pct') < 0) | (F.col('temp_agregado_alta_frontera_cobertura_pct') > 1)
).count()
print(f'rows with temp_agregado_alta_frontera_cobertura_pct out of [0,1]: {bad_temp_coverage_rows}')
if bad_temp_coverage_rows > 0:
    raise ValueError('temp_agregado_alta_frontera_cobertura_pct out of the [0,1] range')

for horizon_days, target_col in [
    (1, 'nivel_rio_t_mas_1d'),
    (3, 'nivel_rio_t_mas_3d'),
    (7, 'nivel_rio_t_mas_7d'),
    (14, 'nivel_rio_t_mas_14d'),
]:
    assert_future_target(horizon_days, target_col)

gold.agg(
    F.min('fecha').alias('inicio'),
    F.max('fecha').alias('fin'),
    F.count('*').alias('rows'),
    F.countDistinct('punto_prediccion').alias('puntos'),
    F.sum(F.when(F.col('nivel_rio_actual_m').isNull(), 1).otherwise(0)).alias('nivel_null_rows'),
    F.sum(F.when(F.col('nivel_rio_t_mas_1d').isNull(), 1).otherwise(0)).alias('target_1d_null_rows'),
).show(truncate=False)